In [ ]:
import time, cv2, numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 3000

H, W = 240, 320
base = rng.integers(60, 200, (H, W), dtype=np.uint8)
base = cv2.GaussianBlur(base, (0, 0), 2)
base = cv2.normalize(base, None, 0, 255, cv2.NORM_MINMAX)
for _ in range(90):
    x, y, r = rng.integers(20, W-20), rng.integers(20, H-20), rng.integers(3, 8)
    cv2.circle(base, (int(x), int(y)), int(r), int(rng.integers(0, 255)), -1)

def frame(dy, ruido=10):
    M = np.float32([[1, 0, 0], [0, 1, dy]])
    f = cv2.warpAffine(base, M, (W, H), borderMode=cv2.BORDER_REFLECT).astype(float)
    return np.clip(f + rng.normal(0, ruido, f.shape), 0, 255).astype(np.uint8)

def medir_vy(f0, f1, dt_h):
    p0 = cv2.goodFeaturesToTrack(f0, maxCorners=200, qualityLevel=0.01, minDistance=8)
    if p0 is None: return 0.0
    p1, st, err = cv2.calcOpticalFlowPyrLK(f0, f1, p0, None, winSize=(21, 21), maxLevel=3)
    ok = st.flatten() == 1
    if ok.sum() < 10: return 0.0
    return float(np.median((p1[ok] - p0[ok]).reshape(-1, 2)[:, 1]) / dt_h)

# velocidade atual (v2) e aceleração reais; v1 = velocidade 1 hora antes
v2_real = np.clip(rng.gamma(1.2, 2.0, n), 0, 15)
a_real = rng.normal(0.2, 0.8, n)
v1_real = np.clip(v2_real - a_real, 0, 15)
a_real = v2_real - v1_real

t0 = time.time()
v2_med, acel_med = [], []
for v1, v2 in zip(v1_real, v2_real):
    f0 = frame(0)
    f1 = frame(v1 + rng.normal(0, 0.3))
    f2 = frame(v1 + v2 + rng.normal(0, 0.3))
    m1 = medir_vy(f0, f1, 1.0)
    m2 = medir_vy(f1, f2, 1.0)
    v2_med.append(m2); acel_med.append(m2 - m1)
v2_med, acel_med = np.array(v2_med), np.array(acel_med)
print("tempo:", round(time.time()-t0, 1), "s")
print("erro velocidade:", np.abs(v2_med - v2_real).mean().round(3), "px/h | erro aceleração:", np.abs(acel_med - a_real).mean().round(3), "px/h²")

df = pd.DataFrame({
    "chuva_24h": rng.gamma(2, 25, n),
    "chuva_72h": rng.gamma(2, 50, n),
    "umidade_solo": rng.uniform(15, 100, n),
    "inclinacao": rng.gamma(1.5, 1.0, n),
    "fluxo_optico": v2_med,
    "aceleracao": acel_med,
    "tem_lona": rng.integers(0, 2, n),
})
score = (0.012*df.chuva_24h + 0.006*df.chuva_72h + 0.015*df.umidade_solo
         + 0.35*df.inclinacao + 0.25*v2_real + 0.30*np.maximum(a_real, 0)
         + 0.5*df.tem_lona + rng.normal(0, 0.25, n))
df["risco"] = pd.cut(score, [-np.inf, np.quantile(score, .55), np.quantile(score, .80),
                             np.quantile(score, .94), np.inf],
                     labels=[0, 1, 2, 3]).astype(int)
print(df.risco.value_counts().sort_index())

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df.drop(columns="risco"), df["risco"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
from sklearn.metrics import recall_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

modelos = {
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "RegLogistica": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
}
for nome, m in modelos.items():
    m.fit(X_tr, y_tr)
    p = m.predict(X_te)
    print(f"{nome}: recall(macro)={recall_score(y_te, p, average='macro'):.3f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred = modelos["XGBoost"].predict(X_te)
print(classification_report(y_te, pred, target_names=["verde","amarelo","laranja","vermelho"]))
print(confusion_matrix(y_te, pred))

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

cal = CalibratedClassifierCV(
    XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42),
    method="isotonic", cv=3)
cal.fit(X_tr, y_tr)
proba = cal.predict_proba(X_te)

pred_padrao = proba.argmax(axis=1)
pred_assim = pred_padrao.copy()
pred_assim[proba[:, 3] > 0.30] = 3      # alerta vermelho com apenas 30% de probabilidade

for nome, p in [("padrao", pred_padrao), ("limiar 30%", pred_assim)]:
    rec = recall_score(y_te == 3, p == 3)
    fn = ((y_te.values == 3) & (p != 3)).sum()
    print(f"{nome}: recall vermelho={rec:.3f}, falsos negativos={fn}")

In [ ]:
imp = pd.Series(modelos["XGBoost"].feature_importances_, index=X.columns).sort_values()
print(imp.round(3))
imp.plot(kind="barh", title="Importância das variáveis (XGBoost)")A

In [ ]:
import joblib
joblib.dump({"modelo": cal, "colunas": list(X.columns), "limiar_critico": 0.30},
            "nucleo_de_risco.joblib")

pacote = joblib.load("nucleo_de_risco.joblib")
NIVEIS = ["verde", "amarelo", "laranja", "vermelho"]

def prever(leitura):
    x = pd.DataFrame([leitura])[pacote["colunas"]]
    p = pacote["modelo"].predict_proba(x)[0]
    nivel = int(p.argmax())
    if p[3] > pacote["limiar_critico"]:
        nivel = 3
    return {"nivel": NIVEIS[nivel], "confianca": round(float(p[nivel])*100, 1),
            "prob_vermelho": round(float(p[3])*100, 1)}

print(prever({"chuva_24h": 120, "chuva_72h": 250, "umidade_solo": 90, "inclinacao": 4,
              "fluxo_optico": 9, "aceleracao": 2, "tem_lona": 1}))
print(prever({"chuva_24h": 5, "chuva_72h": 20, "umidade_solo": 30, "inclinacao": 0.5,
              "fluxo_optico": 0.5, "aceleracao": 0, "tem_lona": 0}))